# 🔭 Astro-Mühendislik ve Fiziksel Modelleme Akademisi
## Ötegezegen Transit Analizi Atölye Programı

| Bilgi | Değer |
|:------|:------|
| **Hedef Sistem** | WASP-126 b |
| **Süre** | 3 × 60 dakika |
| **Seviye** | Lise (İleri Düzey) |

**İçerik:**
1. Transit Fiziği ve Geometrik Optik
2. Limb Darkening ve Sinyal İşleme
3. Kepler Yasaları ve Fiziksel Karakterizasyon

> ▶️ Hücreleri sırayla çalıştırmak için `Shift + Enter` tuşlarını kullanın.


## ⚙️ Kurulum ve Genel Ayarlar

Aşağıdaki hücre tüm gerekli kütüphaneleri yükler, grafik stilini ve fiziksel sabitleri tanımlar. **İlk olarak bu hücreyi çalıştırın.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.gridspec import GridSpec
from scipy.signal import savgol_filter
from scipy.optimize import minimize_scalar
import warnings
warnings.filterwarnings('ignore')

# Grafik stili (koyu tema)
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor':   '#0d1117',
    'axes.edgecolor':   '#c9d1d9',
    'axes.labelcolor':  '#c9d1d9',
    'text.color':       '#c9d1d9',
    'xtick.color':      '#8b949e',
    'ytick.color':      '#8b949e',
    'grid.color':       '#21262d',
    'font.size':        11,
    'font.family':      'sans-serif',
})

# Fiziksel Sabitler
R_SUN_KM     = 6.957e5       # Güneş yarıçapı (km)
R_JUPITER_KM = 7.1492e4      # Jüpiter yarıçapı (km)
AU_KM        = 1.496e8       # 1 AU (km)
G            = 6.674e-11     # Gravitasyon sabiti (m³/kg/s²)
M_SUN_KG     = 1.989e30      # Güneş kütlesi (kg)

print("✅ Tüm kütüphaneler ve sabitler yüklendi.")


---
# 🛰️ Bölüm 1: Matematiksel Modelleme — Transit Fiziği (0-60 dk)

## Temel Kavram

Bir gezegen yıldızının önünden geçtiğinde kapattığı alan, yıldızın toplam alanına oranı **transit derinliğini** verir:

$$\delta = \frac{\Delta F}{F_0} = \left(\frac{R_p}{R_\star}\right)^2$$

Bu basit denklem, ışık eğrisindeki düşüşten gezegenin **boyutunu** doğrudan hesaplamamızı sağlar.


### 1.1 Transit Derinliği Fonksiyonu

In [ ]:
def hesapla_transit_derinligi(R_p, R_s):
    """
    Transit derinliği (δ) hesabı.
    δ = (R_p / R_s)²
    """
    return (R_p / R_s) ** 2

# Test
R_s = 1.0    # Yıldız yarıçapı (normalize)
R_p = 0.1    # Gezegen yarıçapı (Jüpiter ölçeği)

depth = hesapla_transit_derinligi(R_p, R_s)
print(f"Yıldız Yarıçapı (R★)  : {R_s}")
print(f"Gezegen Yarıçapı (Rp) : {R_p}")
print(f"Transit Derinliği (δ) : {depth:.4f}  →  %{depth*100:.2f}")
print(f"Rp / R★ oranı         : {R_p/R_s:.2f}")


### 1.2 Transit Geometrisi Simülasyonu

Aşağıdaki hücre, gezegenin yıldız önündeki 7 farklı konumunu ve karşılık gelen ışık eğrisini çizer.

In [ ]:
def kaplanan_oran(d, R_p, R_s):
    """İki dairenin kesişim alanını hesaplar."""
    if d >= R_s + R_p:
        return 0.0
    if d + R_p <= R_s:
        return (R_p / R_s) ** 2
    if d + R_s <= R_p:
        return 1.0
    r1, r2 = R_s, R_p
    part1 = r1**2 * np.arccos((d**2 + r1**2 - r2**2) / (2 * d * r1))
    part2 = r2**2 * np.arccos((d**2 + r2**2 - r1**2) / (2 * d * r2))
    part3 = 0.5 * np.sqrt((-d+r1+r2) * (d+r1-r2) * (d-r1+r2) * (d+r1+r2))
    return (part1 + part2 - part3) / (np.pi * R_s**2)

R_s, R_p = 1.0, 0.1
depth = hesapla_transit_derinligi(R_p, R_s)

pozisyonlar = np.array([-1.3, -0.8, -0.3, 0.0, 0.3, 0.8, 1.3])
etiketler   = ['Dış (I)', 'Giriş', 'İç-I', 'Merkez', 'İç-II', 'Çıkış', 'Dış (II)']

x_yol = np.linspace(-1.5, 1.5, 1000)
flux  = np.array([1.0 - kaplanan_oran(abs(x), R_p, R_s) for x in x_yol])

fig = plt.figure(figsize=(16, 10))
gs  = GridSpec(2, 1, height_ratios=[1.2, 1], hspace=0.3)

# ÜST: Transit Geometrisi
ax_top = fig.add_subplot(gs[0])
theta  = np.linspace(0, 2*np.pi, 200)
for i in range(20, 0, -1):
    r = R_s * (i / 20)
    daire = plt.Circle((0, 0), r, color=plt.cm.YlOrRd(0.4 + 0.6*(1-(i/20)**0.5)),
                       alpha=0.8, zorder=1)
    ax_top.add_patch(daire)

renkler = plt.cm.cool(np.linspace(0.2, 0.9, len(pozisyonlar)))
for idx, (pos, etiket) in enumerate(zip(pozisyonlar, etiketler)):
    gezegen = plt.Circle((pos, 0), R_p, color=renkler[idx], ec='white', lw=0.8, zorder=10, alpha=0.85)
    ax_top.add_patch(gezegen)
    ax_top.annotate(etiket, (pos, R_p+0.08), fontsize=7, ha='center', color=renkler[idx], fontweight='bold')

ax_top.set_xlim(-1.6, 1.6); ax_top.set_ylim(-0.6, 0.6)
ax_top.set_aspect('equal')
ax_top.set_title('Transit Geometrisi — Gezegenin Yıldız Önündeki Yolculuğu', fontsize=14, fontweight='bold', pad=15)
ax_top.set_xlabel('Konum (R★ biriminde)'); ax_top.grid(False)

# ALT: Işık Eğrisi
ax_bot = fig.add_subplot(gs[1])
ax_bot.plot(x_yol, flux, color='#58a6ff', lw=2, zorder=5)
ax_bot.fill_between(x_yol, flux, 1.0, alpha=0.15, color='#58a6ff')
for idx, pos in enumerate(pozisyonlar):
    f_val = 1.0 - kaplanan_oran(abs(pos), R_p, R_s)
    ax_bot.plot(pos, f_val, 'o', color=renkler[idx], ms=8, zorder=10, mec='white', mew=0.8)
ax_bot.axhline(1.0, color='#8b949e', ls='--', lw=0.8, label='Baz Çizgisi (F₀ = 1)')
ax_bot.axhline(1.0-depth, color='#f85149', ls='--', lw=0.8, label=f'Transit Tabanı (δ = {depth:.4f})')
ax_bot.annotate(f'δ = (Rp/R★)² = ({R_p}/{R_s})² = {depth:.4f}',
                xy=(0, 1-depth), xytext=(0.8, 1-depth-0.003), fontsize=10, color='#f85149',
                fontweight='bold', arrowprops=dict(arrowstyle='->', color='#f85149', lw=1.5))
ax_bot.set_xlabel('Konum (R★ biriminde)'); ax_bot.set_ylabel('Normalize Akı (F / F₀)')
ax_bot.set_title('Transit Işık Eğrisi — Geometrik Model', fontsize=13, fontweight='bold')
ax_bot.legend(loc='lower right', fontsize=9); ax_bot.set_ylim(1-depth*3, 1.001); ax_bot.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


### 1.3 Öğrenci Problemi: Gezegen Boyutu vs Transit Derinliği

In [ ]:
gezegenler = {
    'Dünya benzeri': 0.009, 'Süper-Dünya': 0.02, 'Neptün benzeri': 0.035,
    'Satürn benzeri': 0.083, 'Jüpiter benzeri': 0.10, 'Sıcak Jüpiter': 0.15,
}

print(f"{'Gezegen Tipi':<20} {'Rp/R★':<10} {'δ (%)':<12} {'SNR Notu'}")
print("─" * 55)
for isim, oran in gezegenler.items():
    d = hesapla_transit_derinligi(oran, 1.0)
    snr = "Zor" if d < 0.001 else ("Orta" if d < 0.005 else "Kolay")
    print(f"{isim:<20} {oran:<10.3f} {d*100:<12.4f} {snr}")

fig, ax = plt.subplots(figsize=(10, 6))
oranlar = np.linspace(0.005, 0.2, 200)
ax.plot(oranlar, (oranlar**2)*100, color='#58a6ff', lw=2.5)
ax.fill_between(oranlar, (oranlar**2)*100, alpha=0.1, color='#58a6ff')

renkler_g = ['#7ee787', '#3fb950', '#f0883e', '#d29922', '#f85149', '#da3633']
for (isim, oran), renk in zip(gezegenler.items(), renkler_g):
    ax.plot(oran, (oran**2)*100, 'o', color=renk, ms=10, zorder=10, mec='white', mew=1.5)
    ax.annotate(isim, (oran, (oran**2)*100), textcoords="offset points",
                xytext=(10, 5), fontsize=8, color=renk, fontweight='bold')

ax.axhline(0.01, color='#f85149', ls='--', lw=1, alpha=0.7, label='Kepler Tespit Sınırı (~100 ppm)')
ax.set_xlabel('Rp / R★', fontsize=12); ax.set_ylabel('Transit Derinliği δ (%)', fontsize=12)
ax.set_title('δ = (Rp/R★)²', fontsize=13, fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3); ax.set_yscale('log')
plt.tight_layout(); plt.show()


---
# 🤖 Bölüm 2: İleri Sinyal İşleme — Limb Darkening (60-120 dk)

## Neden Yıldız Homojen Parlaklıkta Değildir?

Yıldızın **merkezi** kenarlarından daha parlaktır. Buna **Limb Darkening** (kenar kararması) denir.

**Kuadratik Model:**

$$\frac{I(\mu)}{I(1)} = 1 - u_1(1-\mu) - u_2(1-\mu)^2$$

Burada $\mu = \cos\theta = \sqrt{1-r^2}$ ve $r$ yıldız diskindeki normalize yarıçaptır.


### 2.1 Limb Darkening Profili ve Yıldız Diski

In [ ]:
def limb_darkening_profili(r, u1=0.4, u2=0.2):
    """Kuadratik Limb Darkening: I(r)/I(0) = 1 - u1*(1-μ) - u2*(1-μ)²"""
    r = np.clip(r, 0, 0.999)
    mu = np.sqrt(1.0 - r**2)
    return 1.0 - u1*(1-mu) - u2*(1-mu)**2


### 2.2 Transit Modeli (Mandel & Agol Yaklaşımı)

In [ ]:
def mandel_agol_transit(t, T0, P, Rp_Rs, a_Rs, inc_deg, u1=0.4, u2=0.2):
    """Basitleştirilmiş transit modeli — kuadratik LD dahil."""
    inc = np.radians(inc_deg)
    faz = 2.0 * np.pi * (t - T0) / P
    x = a_Rs * np.sin(faz)
    y = a_Rs * np.cos(faz) * np.cos(inc)
    z = a_Rs * np.cos(faz) * np.sin(inc)
    d = np.sqrt(x**2 + y**2)
    p = Rp_Rs
    flux = np.ones_like(t)

    for i in range(len(t)):
        if z[i] < 0: continue
        di = d[i]
        if di >= 1.0 + p: continue
        n_ring = 100
        r_rings = np.linspace(0, 1.0, n_ring + 1)
        toplam_isik = 0.0; kaplanan_isik = 0.0
        for j in range(n_ring):
            r_ic, r_dis = r_rings[j], r_rings[j+1]
            r_ort = (r_ic + r_dis) / 2.0
            alan  = np.pi * (r_dis**2 - r_ic**2)
            yogunluk = limb_darkening_profili(r_ort, u1, u2)
            toplam_isik += yogunluk * alan
            if di + p <= r_ic or di - p >= r_dis: continue
            if abs(di - r_ort) < p:
                theta_kap = 2.0 * np.arcsin(np.clip(p / max(r_ort, 0.01), 0, 1))
                kaplanan_isik += yogunluk * alan * theta_kap / (2*np.pi)
        if toplam_isik > 0:
            flux[i] = 1.0 - kaplanan_isik / toplam_isik
    return flux

print("✅ Transit modeli hazır.")


### 2.3 Kapsamlı Limb Darkening Görselleştirmesi

In [ ]:
fig = plt.figure(figsize=(16, 12))
gs = GridSpec(2, 2, hspace=0.35, wspace=0.3)

# Panel 1: LD Profilleri
ax1 = fig.add_subplot(gs[0, 0])
r = np.linspace(0, 1, 500)
modeller = {'Sabit (u₁=0, u₂=0)': (0,0), 'Lineer (u₁=0.6)': (0.6,0),
            'Kuadratik (u₁=0.4, u₂=0.2)': (0.4,0.2), 'Güçlü LD (u₁=0.6, u₂=0.3)': (0.6,0.3)}
for (isim, (u1,u2)), renk in zip(modeller.items(), ['#8b949e','#58a6ff','#f0883e','#f85149']):
    ax1.plot(r, limb_darkening_profili(r, u1, u2), color=renk, lw=2, label=isim)
ax1.set_xlabel('Normalize Yarıçap (r/R★)'); ax1.set_ylabel('I(r)/I(0)')
ax1.set_title('Limb Darkening Profilleri', fontsize=12, fontweight='bold')
ax1.legend(fontsize=8, loc='lower left'); ax1.grid(True, alpha=0.3)

# Panel 2: 2D Yıldız Diski
ax2 = fig.add_subplot(gs[0, 1])
x_g = np.linspace(-1, 1, 500); X, Y = np.meshgrid(x_g, x_g); R_g = np.sqrt(X**2+Y**2)
I_g = np.where(R_g <= 1, limb_darkening_profili(R_g, 0.4, 0.2), 0)
ax2.imshow(I_g, extent=[-1,1,-1,1], cmap='YlOrRd', origin='lower', vmin=0, vmax=1.1)
ax2.add_patch(plt.Circle((0.3, 0.1), 0.1, color='black', ec='white', lw=1, zorder=10))
ax2.set_xlim(-1.2, 1.2); ax2.set_ylim(-1.2, 1.2); ax2.set_aspect('equal')
ax2.set_title('Yıldız Diski + Gezegen\n(u₁=0.4, u₂=0.2)', fontsize=11, fontweight='bold')

# Panel 3: LD'li vs LD'siz Transit
ax3 = fig.add_subplot(gs[1, 0])
P, T0, Rp_Rs, a_Rs, inc = 3.2888, 0.0, 0.078, 7.8, 87.8
t_tr = np.linspace(-0.15, 0.15, 800)
flux_flat = mandel_agol_transit(t_tr, T0, P, Rp_Rs, a_Rs, inc, 0, 0)
flux_ld   = mandel_agol_transit(t_tr, T0, P, Rp_Rs, a_Rs, inc, 0.4, 0.2)
ax3.plot(t_tr*24, flux_flat, color='#8b949e', lw=2, ls='--', label='LD Yok', alpha=0.8)
ax3.plot(t_tr*24, flux_ld, color='#f0883e', lw=2.5, label='Kuadratik LD')
ax3.fill_between(t_tr*24, flux_flat, flux_ld, alpha=0.2, color='#f0883e')
ax3.set_xlabel('Saat'); ax3.set_ylabel('Akı')
ax3.set_title('Limb Darkening Etkisi', fontsize=12, fontweight='bold'); ax3.legend(fontsize=9); ax3.grid(True, alpha=0.3)

# Panel 4: Savitzky-Golay
ax4 = fig.add_subplot(gs[1, 1])
np.random.seed(42)
t_d = np.linspace(-0.2, 0.2, 600)
flux_temiz = mandel_agol_transit(t_d, T0, P, Rp_Rs, a_Rs, inc, 0.4, 0.2)
flux_ham = flux_temiz + np.random.normal(0, 0.0015, len(t_d))
flux_sg = savgol_filter(flux_ham, 31, 3)
ax4.scatter(t_d*24, flux_ham, s=2, alpha=0.4, color='#8b949e', label='Ham Veri')
ax4.plot(t_d*24, flux_sg, color='#f85149', lw=2, label='SG Filtresi')
ax4.plot(t_d*24, flux_temiz, color='#7ee787', lw=1.5, ls='--', alpha=0.7, label='Gerçek Sinyal')
ax4.set_xlabel('Saat'); ax4.set_ylabel('Akı')
ax4.set_title('Savitzky-Golay Filtresi', fontsize=12, fontweight='bold'); ax4.legend(fontsize=9, loc='lower right'); ax4.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

rms_ham = np.std(flux_ham - flux_temiz)
rms_sg  = np.std(flux_sg  - flux_temiz)
print(f"\nHam RMS: {rms_ham*1e6:.0f} ppm  |  SG RMS: {rms_sg*1e6:.0f} ppm  |  İyileşme: {rms_ham/rms_sg:.1f}x")


---
# 📏 Bölüm 3: Fiziksel Karakterizasyon ve Kepler Yasaları (120-180 dk)

## Kepler 3. Yasası

$$a^3 = \frac{G \, M_\star \, P^2}{4\pi^2}$$

Basitleştirilmiş form:

$$a \;(\mathrm{AU}) \approx \left[ M_\star \;(M_\odot) \times P^2 \;(\mathrm{yıl}) \right]^{1/3}$$


### 3.1 WASP-126 b Tam Karakterizasyon

In [ ]:
# ── WASP-126 Parametreleri ──
m_s   = 1.12;  r_s  = 1.27;  T_eff = 5800
P_gun = 3.2888; Rp_Rs = 0.078  # NASA Exoplanet Archive

# ── Gezegen Yarıçapı ──
Rp_Rsun = Rp_Rs * r_s
Rp_Rjup = Rp_Rsun * R_SUN_KM / R_JUPITER_KM
Rp_km   = Rp_Rsun * R_SUN_KM

# ── Yörünge Yarıçapı (Kepler 3) ──
P_s  = P_gun * 86400.0
a_m  = (G * m_s * M_SUN_KG * P_s**2 / (4*np.pi**2))**(1/3)
a_au = a_m / (AU_KM * 1e3)
a_km = a_m / 1e3

# ── Denge Sıcaklığı ──
albedo = 0.3
T_eq = T_eff * np.sqrt(r_s * R_SUN_KM / (2*a_km)) * (1-albedo)**0.25

# ── Yaşanabilirlik Bölgesi ──
# Lüminozite temelli HZ (Kopparapu et al. 2013)
L_star = 10**0.145  # L☉, NASA Exoplanet Archive: log(L)≈0.145
hz_ic  = 0.95 * np.sqrt(L_star)
hz_dis = 1.67 * np.sqrt(L_star)
hz_icinde = hz_ic < a_au < hz_dis

print("═══ WASP-126 b — Fiziksel Parametreler ═══")
print(f"Gezegen Yarıçapı  : {Rp_Rjup:.2f} R_J  ({Rp_km:.0f} km)")
print(f"Yörünge Yarıçapı  : {a_au:.4f} AU  ({a_km:.0f} km)")
print(f"Denge Sıcaklığı   : {T_eq:.0f} K")
print(f"HZ İç Sınır       : {hz_ic:.3f} AU")
print(f"HZ Dış Sınır      : {hz_dis:.3f} AU")
print(f"HZ Durumu          : {'İÇİNDE ✓' if hz_icinde else 'DIŞINDA ✗'}")
sinif = "Sıcak Gaz Devi (düşük yoğunluklu)" if Rp_Rjup > 0.8 else ("Sıcak Satürn" if Rp_Rjup > 0.3 else "Sıcak Neptün")
print(f"Sınıf              : {sinif}")


### 3.2 Kapsamlı Görselleştirme

In [ ]:
fig = plt.figure(figsize=(18, 14))
gs = GridSpec(2, 2, hspace=0.35, wspace=0.3)

# Panel 1: Kepler 3. Yasa
ax1 = fig.add_subplot(gs[0, 0])
P_r = np.linspace(0.5, 400, 500)
a_r = (m_s * (P_r/365.25)**2)**(1/3)
ax1.plot(P_r, a_r, color='#58a6ff', lw=2)
ax1.plot(P_gun, a_au, '*', color='#f0883e', ms=20, zorder=10, mec='white', mew=1.5,
         label=f'WASP-126 b\n(P={P_gun}d, a={a_au:.3f}AU)')
for isim, (pg, ag) in {'Merkür':(87.97,0.387),'Venüs':(224.7,0.723),
                         'Dünya':(365.25,1.0),'Mars':(687,1.524)}.items():
    if pg <= 400:
        ax1.plot(pg, ag, 'o', color='#7ee787', ms=8, mec='white', mew=1)
        ax1.annotate(isim, (pg, ag), textcoords="offset points", xytext=(8,5), fontsize=8, color='#7ee787')
ax1.axhspan(hz_ic, hz_dis, alpha=0.15, color='#3fb950', label='Yaşanabilirlik Bölgesi')
ax1.set_xlabel('Periyot (gün)'); ax1.set_ylabel('a (AU)')
ax1.set_title('Kepler 3. Yasası: P² ∝ a³', fontsize=12, fontweight='bold')
ax1.legend(fontsize=9); ax1.grid(True, alpha=0.3); ax1.set_xscale('log'); ax1.set_yscale('log')

# Panel 2: BLS Periodogram (Temsili Simülasyon)
ax2 = fig.add_subplot(gs[0, 1])
tp = np.linspace(1, 10, 5000); np.random.seed(123)
bls = np.random.exponential(0.5, len(tp))
idx_g = np.argmin(np.abs(tp - P_gun))
bls[idx_g-5:idx_g+5] += [2,5,10,18,25,25,18,10,5,2]
for h in [P_gun/2, P_gun*2]:
    idx_h = np.argmin(np.abs(tp - h))
    bls[max(0,idx_h-3):idx_h+3] += [1,3,6,6,3,1]
ax2.plot(tp, bls, color='#58a6ff', lw=0.8, alpha=0.8)
ax2.axvline(P_gun, color='#f85149', ls='--', lw=1.5, label=f'P = {P_gun:.4f} gün')
ax2.axhline(7, color='#f0883e', ls=':', lw=1, label='SNR 7σ eşiği', alpha=0.7)
ax2.set_xlabel('Test Periyodu (gün)'); ax2.set_ylabel('BLS Gücü (σ)')
ax2.set_title('BLS Periodogram', fontsize=12, fontweight='bold'); ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)

# Panel 3: Yörünge Diyagramı
ax3 = fig.add_subplot(gs[1, 0])
th = np.linspace(0, 2*np.pi, 500)
ax3.plot(a_au*np.cos(th), a_au*np.sin(th), color='#58a6ff', lw=1.5, ls='--', alpha=0.6)
ax3.add_patch(plt.Circle((0,0), r_s*R_SUN_KM/AU_KM, color='#f0883e', alpha=0.9, zorder=5))
gx, gy = a_au*np.cos(np.pi/4), a_au*np.sin(np.pi/4)
ax3.add_patch(plt.Circle((gx, gy), 0.003, color='#7ee787', zorder=10))
ax3.annotate('WASP-126 b', (gx, gy), textcoords="offset points", xytext=(10,10), fontsize=10, color='#7ee787', fontweight='bold')
ax3.add_patch(plt.Circle((0,0), hz_ic, fill=False, color='#3fb950', ls='--', lw=1, alpha=0.5))
ax3.add_patch(plt.Circle((0,0), hz_dis, fill=False, color='#3fb950', ls='--', lw=1, alpha=0.5))
ax3.set_xlim(-0.15, 0.15); ax3.set_ylim(-0.15, 0.15); ax3.set_aspect('equal')
ax3.set_xlabel('x (AU)'); ax3.set_ylabel('y (AU)')
ax3.set_title('Yörünge Diyagramı', fontsize=12, fontweight='bold'); ax3.grid(True, alpha=0.2)

# Panel 4: Kimlik Kartı
ax4 = fig.add_subplot(gs[1, 1]); ax4.axis('off')
kart = f"""
╔═════════════════════════════════════╗
║     GEZEGEN KİMLİK KARTI           ║
║     WASP-126 b                      ║
╠═════════════════════════════════════╣
║  Yarıçap  : {Rp_Rjup:.2f} R_J ({Rp_km:.0f} km)║
║  Periyot  : {P_gun:.4f} gün           ║
║  Uzaklık  : {a_au:.4f} AU             ║
║  Sıcaklık : {T_eq:.0f} K                ║
║  Sınıf    : {sinif:<20}  ║
║  HZ       : {'İÇİNDE' if hz_icinde else 'DIŞINDA':<20}  ║
║  SNR      : > 7σ  ✓                ║
╠═════════════════════════════════════╣
║  DURUM: TEMSİLİ DOĞRULAMA ✓          ║
╚═════════════════════════════════════╝
"""
ax4.text(0.05, 0.95, kart, fontsize=10, fontfamily='monospace', va='top', color='#7ee787',
         bbox=dict(boxstyle='round', facecolor='#161b22', edgecolor='#30363d'))
ax4.set_title('Teknik Karakterizasyon Raporu', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()


### 3.3 Teknik Kontrol Listesi — Keşif Onay Matrisi

In [ ]:
kontroller = [
    ('SNR',                '> 7.0 σ',        True,  'Sinyal gürültüden yeterince güçlü.'),
    ('V-Shape Testi',      'Taban düz (U)',   True,  'U şekli → Gezegen. V → Çift yıldız.'),
    ('Secondary Eclipse',  'İkinci çukur yok',True,  'Çift yıldız senaryosu elendi.'),
    ('Duration Check',     'Süre uyumlu',     True,  f'P={P_gun:.2f}d, a={a_au:.3f}AU ile tutarlı.'),
    ('Periyot Stabilitesi','±0.001 gün',      True,  'Ardışık transitler arası sabit.'),
]

print("══════════════════════════════════════════════════════════════════════")
print(f"  {'Kontrol':<22} {'Kriter':<20} {'Sonuç':<12} {'Yorum'}")
print("──────────────────────────────────────────────────────────────────────")
for isim, kriter, ok, yorum in kontroller:
    print(f"  {isim:<22} {kriter:<20} {'✓ Başarılı' if ok else '✗ Başarısız':<12} {yorum}")
print("══════════════════════════════════════════════════════════════════════")
print("  ✅  TÜM KONTROLLER BAŞARILI → TEMSİLİ DOĞRULAMA TAMAMLANDI (Öğretim Amaçlı)")
print("══════════════════════════════════════════════════════════════════════")


---
## 🎓 Atölye Tamamlandı!

**Kazanımlar:**
- Transit derinliği ile gezegen boyutu hesabı
- Limb darkening modellemesi ve sinyal işleme
- Kepler yasaları ile yörünge ve sıcaklık karakterizasyonu
- Sistematik keşif doğrulama süreci

**Sonraki Adım:** Kendi seçtiğiniz bir TESS/Kepler hedefi için aynı analizi tekrarlayın ve Teknik Karakterizasyon Raporu hazırlayın.
